# ShuffleNetV2 - Individual Baseline Run


In [1]:
import sys
import os

import torch

sys.path.append(os.path.abspath("../src"))

from models import ShuffleNetV2
from data import prepare_full_dataframe, prepare_data
from train import run_training_pipeline, run_smoke_test
from utils import (
    get_device,
    print_model_overrides,
)
import config

print(f"Python:         {sys.executable}")
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")

Python:         c:\Users\profb\PROJECTS\cxr-model-benchmark\.venv\Scripts\python.exe
PyTorch:        2.11.0+cu128
CUDA available: True
GPU:            NVIDIA GeForce RTX 5090 Laptop GPU


In [2]:
dataset_path = config.DATASET_PATH
print("Dataset location:", dataset_path)

metadata_file = os.path.join(dataset_path, "Data_Entry_2017.csv")
df = prepare_full_dataframe(metadata_file, dataset_path)

print(f"Total images:    {len(df)}")
print(f"Unique patients: {df['Patient ID'].nunique()}")

Dataset location: C:\Users\profb\PROJECTS\datasets\NIH_Chest_X-Rays
Total images:    112120
Unique patients: 30805


In [3]:
print(df["split"].value_counts())

split
train    78934
val      16874
test     16312
Name: count, dtype: int64


In [4]:
train_loader, val_loader, test_loader = prepare_data(df, model_name="ShuffleNetV2")
print("Train transforms:", train_loader.dataset.transform)
device = get_device()

Train transforms: Compose(
    Resize(size=(256, 256), interpolation=bilinear, max_size=None, antialias=True)
    RandomRotation(degrees=[-12.0, 12.0], interpolation=nearest, expand=False, fill=0)
    RandomAffine(degrees=[0.0, 0.0], translate=(0.05, 0.05), scale=(0.94, 1.06))
    ColorJitter(brightness=(0.88, 1.12), contrast=(0.88, 1.12), saturation=None, hue=None)
    ToTensor()
    Normalize(mean=[0.5], std=[0.25])
)
Using CUDA (GPU)


## Smoke Test


In [5]:
run_smoke_test(
    model_name="ShuffleNetV2",
    model_builder=lambda: ShuffleNetV2(num_classes=2, in_channels=1),
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    device=device,
    live_plot=False,
)


=== Smoke test: ShuffleNetV2 (epochs=1, patience=1, batches=1) ===

=== Training ShuffleNetV2 ===

=== Run Configuration ===
Model: ShuffleNetV2
Training: epochs=1, patience=999, batch_size=256, image_size=256, seed=42
Precision/Memory: amp_enabled=True, amp_dtype=bf16, channels_last(global=True, override=None, effective=True)
Optimization: layerwise_lr_enabled=True, uses_param_groups=True, freeze_backbone_enabled=True, freeze_backbone_epochs=2, lr=0.0001, backbone_lr=7.5e-05, head_lr=0.0003, weight_decay=5e-05, label_smoothing=0.03, loss_type=cross_entropy, loss_class_weights=None, focal_gamma=2
Scheduler: enabled=True, type=warmup_cosine, start_epoch=1, warmup_epochs=0, warmup_start_factor=0.4, cosine_t_max=12, min_lr=1e-06, steps_per_epoch=1
Checkpoint Resume: False
Model Overrides: {'lr': 0.0001, 'backbone_lr': 7.5e-05, 'head_lr': 0.0003, 'patience': 999, 'freeze_backbone_epochs': 2, 'channels_last': None, 'weight_decay': None, 'label_smoothing': None, 'loss_type': None, 'loss_cla

Train Epoch 1:   0%|          | 0/1 [00:00<?, ?it/s]

Val Epoch 1:   0%|          | 0/1 [00:00<?, ?it/s]


Epoch 1/1
  Train Loss: 0.6937
  Train Acc:  0.4609
  Val Loss:   0.6921
  Val Acc:    0.5352
  Val AUPRC:  0.4880
  Val Recall: 0.0000 | Val F1: 0.0000
  → Best epoch so far (saved model)
  Backbone LR: 7.37393e-05 | Head LR: 0.000294906
  Optimizer: AdamW | Weight Decay: 5e-05 | No-Decay Groups: 2


Test:   0%|          | 0/1 [00:00<?, ?it/s]

({'model': 'ShuffleNetV2',
  'epochs': 1,
  'batch_size': 256,
  'image_size': 256,
  'test_loss': 0.693246603012085,
  'accuracy': 0.48046875,
  'precision': 0.0,
  'recall': 0.0,
  'f1': 0.0,
  'auprc': 0.5925235213201734},
 {'train_loss': [0.6936940550804138],
  'train_acc': [0.4609375],
  'val_loss': [0.6920831203460693],
  'val_acc': [0.53515625],
  'val_precision': [0.0],
  'val_recall': [0.0],
  'val_f1': [0.0],
  'val_auprc': [0.48802237831486833],
  'lr': [0.00029490591103021566],
  'lr_backbone': [7.373925557269551e-05],
  'lr_head': [0.00029490591103021566],
  'backbone_frozen': [True],
  'best_epoch': 1})

## Training


### Function Definitions

In [6]:
run_results = []
baseline_auprc = 0.6407

from notebook_experiment_runner import (
    cleanup_training_artifacts,
    print_planned_run_configuration,
    run_seed_experiment,
)

In [7]:
# Optional per-run knobs for this notebook execution
seed_bank = tuple(config.TUNING_OVERRIDES.get("ShuffleNetV2", {}).get("seed_bank", [16, 32, 64]))
epoch_log_file = "../outputs/experiment_outputs/shufflenetv2_epoch_metrics.log"

def run_shufflenet_experiment(seeds=seed_bank, live_plot=True, reset_results=False, reset_epoch_log=True):
    run_seed_experiment(
        run_results=run_results,
        baseline_auprc=baseline_auprc,
        model_builder=lambda: ShuffleNetV2(num_classes=2, in_channels=1),
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device,
        model_name="ShuffleNetV2",
        seeds=seeds,
        live_plot=live_plot,
        reset_results=reset_results,
        epoch_log_file=epoch_log_file,
        reset_epoch_log=reset_epoch_log,
    )

### Run

In [8]:
print_planned_run_configuration(
    model_name="ShuffleNetV2",
    model_builder=lambda: ShuffleNetV2(num_classes=2, in_channels=1),
    device=device,
    resume_from_checkpoint=False,
)


=== Run Configuration ===
Model: ShuffleNetV2
Training: epochs=30, patience=999, batch_size=256, image_size=256, seed=42
Precision/Memory: amp_enabled=True, amp_dtype=bf16, channels_last(global=True, override=None, effective=True)
Optimization: layerwise_lr_enabled=True, uses_param_groups=True, freeze_backbone_enabled=True, freeze_backbone_epochs=2, lr=0.0001, backbone_lr=7.5e-05, head_lr=0.0003, weight_decay=5e-05, label_smoothing=0.03, loss_type=cross_entropy, loss_class_weights=None, focal_gamma=2
Scheduler: enabled=True, type=warmup_cosine, start_epoch=1, warmup_epochs=1, warmup_start_factor=0.4, cosine_t_max=12, min_lr=1e-06, steps_per_epoch=1
Checkpoint Resume: False
Model Overrides: {'lr': 0.0001, 'backbone_lr': 7.5e-05, 'head_lr': 0.0003, 'patience': 999, 'freeze_backbone_epochs': 2, 'channels_last': None, 'weight_decay': None, 'label_smoothing': None, 'loss_type': None, 'loss_class_weights': None, 'focal_gamma': None, 'scheduler_min_lr': None, 'scheduler_start_epoch': None, '

In [9]:
run_shufflenet_experiment()

Epoch log reset: C:\Users\profb\PROJECTS\cxr-model-benchmark\outputs\experiment_outputs\shufflenetv2_epoch_metrics.log


### Seed Training Outputs

### Stacked Snapshots

Output()

#### Run 1 | Seed 16

...

Output()

#### Run 4 | Seed 128

Output()

#### Run 5 | Seed 256

Output()

## Complete Metrics

In [10]:
import pandas as pd

summary_df = pd.DataFrame([
    {
        "Seed": r["seed"],
        "Best Epoch": r["best_epoch"],
        "Best Val AUPRC": f"{r['best_val_auprc']:.4f}" if r['best_val_auprc'] else "N/A",
        "Delta": f"{r['delta']:.4f}" if r['delta'] else "N/A",
    }
    for r in run_results
])

print("\n=== Run Comparison ===")
print(summary_df.to_string(index=False))

if len(run_results) > 1:
    valid_auprc = [r["best_val_auprc"] for r in run_results if r["best_val_auprc"] is not None]
    valid_best_epochs = [r["best_epoch"] for r in run_results if r["best_epoch"] is not None]
    mean_auprc = sum(valid_auprc) / len(valid_auprc)
    mean_delta = sum(r["delta"] for r in run_results if r["delta"] is not None) / len(valid_auprc)
    mean_best_epoch = sum(valid_best_epochs) / len(valid_best_epochs)
    std_auprc = (sum((x - mean_auprc) ** 2 for x in valid_auprc) / len(valid_auprc)) ** 0.5
    print(f"\nMean Best Epoch: {mean_best_epoch:.2f}")
    print(f"Mean Val AUPRC: {mean_auprc:.4f} ({mean_delta:+.4f})")
    print(f"Std Dev: {std_auprc:.4f}")


=== Run Comparison ===
 Seed  Best Epoch Best Val AUPRC  Delta
   16          19         0.7040 0.0633
   32          13         0.7055 0.0648
   64          17         0.7020 0.0613
  128          23         0.7029 0.0622
  256          17         0.7043 0.0636

Mean Best Epoch: 17.80
Mean Val AUPRC: 0.7038 (+0.0631)
Std Dev: 0.0012
